# Data Preparation — Churn Prediction
## HPB Fintech Hackathon 2026

**Input:** Raw features from EDA (`churn_features_raw.csv`)
**Output:** Clean, training-ready dataset (`churn_features_clean.csv`)

### Steps
1. Load and inspect raw features
2. Handle missing values
3. Analyze feature distributions
4. Check correlations and remove redundant features
5. Export clean dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

from pathlib import Path
DATA = Path('../data')

df = pd.read_csv(DATA / 'churn_features_raw.csv')
print(f'Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Periods: {df["period"].value_counts().to_dict()}')
print(f'Churn rate: {df["churned"].mean():.1%}')
print()
df.info()

## 1. Missing Value Analysis

In [ ]:
FEATURES = ['dob', 'tenure_years', 'n_products', 'has_kredit', 'prima_placu',
            'avg_txn_per_month', 'avg_txn_amount', 'txn_trend',
            'avg_balance', 'balance_trend', 'has_contact', 'n_contacts']

missing = df[FEATURES].isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
miss_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct}).query('Missing > 0')
print('Missing values:')
if len(miss_df) > 0:
    print(miss_df.to_string())
else:
    print('  None')

# Imputation strategy:
# - Behavioral features (txn, balance, contact): NaN means no activity -> fill with 0
# - Demographic features (dob, tenure): fill with median
behavioral = ['avg_txn_per_month', 'avg_txn_amount', 'txn_trend',
              'avg_balance', 'balance_trend', 'n_contacts']
demographic = ['dob', 'tenure_years']

for col in behavioral:
    n_filled = df[col].isnull().sum()
    df[col] = df[col].fillna(0)
    if n_filled > 0:
        print(f'  {col}: filled {n_filled:,} NaN with 0')

for col in demographic:
    n_filled = df[col].isnull().sum()
    if n_filled > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  {col}: filled {n_filled:,} NaN with median ({median_val:.1f})')

print(f'\nRemaining NaN: {df[FEATURES].isnull().sum().sum()}')

## 2. Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    ax = axes[i]
    for val, label, color in [(0, 'Active', '#2ecc71'), (1, 'Churned', '#e74c3c')]:
        subset = df.loc[df['churned'] == val, col].dropna()
        if col in ['has_kredit', 'prima_placu', 'has_contact']:
            # Binary feature - show proportions
            counts = df[df['churned'] == val][col].value_counts(normalize=True)
            ax.bar([f'{label}\n0', f'{label}\n1'], [counts.get(0, 0), counts.get(1, 0)],
                   alpha=0.6, color=color, width=0.35,
                   left=[0 if val == 0 else 0.35, 0 if val == 0 else 0.35])
        else:
            clipped = subset.clip(upper=subset.quantile(0.99))
            ax.hist(clipped, bins=30, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(col, fontweight='bold', fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Churn Status', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlation Analysis & Feature Selection

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, vmin=-1, vmax=1, square=True, linewidths=0.5)
ax.set_title('Feature Correlations', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Check for highly correlated pairs (|r| > 0.8)
high_corr = []
for i in range(len(FEATURES)):
    for j in range(i+1, len(FEATURES)):
        r = abs(corr.iloc[i, j])
        if r > 0.8:
            high_corr.append((FEATURES[i], FEATURES[j], r))

if high_corr:
    print('Highly correlated pairs (|r| > 0.8):')
    for f1, f2, r in sorted(high_corr, key=lambda x: -x[2]):
        print(f'  {f1} <-> {f2}: {r:.3f}')
    print('Consider dropping one feature from each pair.')
else:
    print('No feature pairs with |r| > 0.8 - all features retained.')

# Point-biserial correlation with target
target_corr = df[FEATURES].corrwith(df['churned']).abs().sort_values(ascending=False)
print(f'\nCorrelation with churn target:')
print(target_corr.round(4).to_string())

In [ ]:
# ── Final Feature Selection ──
# Drop features with very high mutual correlation (if any)
# Keep all features with meaningful target correlation

FINAL_FEATURES = FEATURES.copy()

# Remove highly correlated pairs - keep the one with higher target correlation
removed = set()
for f1, f2, r in sorted(high_corr, key=lambda x: -x[2]):
    if f1 in removed or f2 in removed:
        continue
    tc1 = abs(df[f1].corr(df['churned']))
    tc2 = abs(df[f2].corr(df['churned']))
    drop = f1 if tc1 < tc2 else f2
    removed.add(drop)
    FINAL_FEATURES.remove(drop)
    print(f'Dropped {drop} (r={r:.3f} with {f1 if drop == f2 else f2}, '
          f'lower target correlation {min(tc1,tc2):.4f})')

if not removed:
    print('All features retained.')

print(f'\nFinal feature set ({len(FINAL_FEATURES)} features):')
for i, f in enumerate(FINAL_FEATURES, 1):
    print(f'  {i:2d}. {f}')

# Export clean dataset
out = df[['IDENTIFIKATOR_KLIJENTA', 'period'] + FINAL_FEATURES + ['churned']].copy()
out_path = DATA / 'churn_features_clean.csv'
out.to_csv(out_path, index=False)
print(f'\nExported: {out_path}')
print(f'  Shape: {out.shape}')
print(f'  Churn rate: {out["churned"].mean():.1%}')